# 提示工程基础 (Prompt Engineering Basics)

> **学习目标**：掌握提示工程的核心概念和基础技巧

---

## 目录

1. [提示工程概述](#1-提示工程概述)
2. [提示的基本结构](#2-提示的基本结构)
3. [提示设计原则](#3-提示设计原则)
4. [Zero-shot Prompting](#4-zero-shot-prompting)
5. [角色提示](#5-角色提示)
6. [输出格式控制](#6-输出格式控制)
7. [实战练习](#7-实战练习)

In [1]:
# 环境设置
import sys
sys.path.append('..')

from src.prompt_templates import PromptTemplate, ChatPromptTemplate, PromptLibrary, Message
import json
import re

## 1. 提示工程概述

### 1.1 什么是提示工程？

**提示工程 (Prompt Engineering)** 是设计和优化输入提示以引导大语言模型产生期望输出的技术。

```
用户意图 → [提示设计] → 结构化Prompt → [LLM] → 期望输出
```

### 1.2 为什么提示工程重要？

| 场景 | 无提示工程 | 有提示工程 |
|------|-----------|------------|
| 数学推理 | 准确率 ~20% | 准确率 ~80% (CoT) |
| 格式化输出 | 格式混乱 | 结构清晰 |
| 复杂任务 | 经常失败 | 稳定完成 |

## 2. 提示的基本结构

一个完整的提示通常包含以下要素：

```
Prompt = 角色设定 + 任务描述 + 上下文 + 示例 + 输出格式 + 约束条件
```

In [2]:
# 提示的基本结构示例
structured_prompt = """
【角色设定】
你是一位资深的Python开发专家，擅长代码审查和优化。

【任务描述】
请分析以下代码的性能问题，并提供优化建议。

【上下文】
这是一个处理大量数据的ETL脚本，每天运行一次。

【代码】
```python
def process_data(data):
    result = []
    for item in data:
        if item not in result:
            result.append(item)
    return result
```

【输出格式】
请按以下格式输出：
1. 问题分析
2. 优化建议
3. 优化后代码

【约束条件】
- 保持代码可读性
- 不使用第三方库
"""

print(structured_prompt)


【角色设定】
你是一位资深的Python开发专家，擅长代码审查和优化。

【任务描述】
请分析以下代码的性能问题，并提供优化建议。

【上下文】
这是一个处理大量数据的ETL脚本，每天运行一次。

【代码】
```python
def process_data(data):
    result = []
    for item in data:
        if item not in result:
            result.append(item)
    return result
```

【输出格式】
请按以下格式输出：
1. 问题分析
2. 优化建议
3. 优化后代码

【约束条件】
- 保持代码可读性
- 不使用第三方库



## 3. 提示设计原则

### 3.1 清晰性原则

In [3]:
# 差的提示 vs 好的提示

bad_prompt = "帮我写代码"

good_prompt = """
请用Python实现一个函数，功能是：
1. 输入：一个整数列表
2. 输出：列表中所有偶数的平方和
3. 要求：使用列表推导式，添加类型注解
"""

print("差的提示：")
print(bad_prompt)
print("\n好的提示：")
print(good_prompt)

差的提示：
帮我写代码

好的提示：

请用Python实现一个函数，功能是：
1. 输入：一个整数列表
2. 输出：列表中所有偶数的平方和
3. 要求：使用列表推导式，添加类型注解



In [4]:
# 3.2 具体性原则

specificity_examples = {
    "模糊": "写得好一点",
    "具体": "使用专业术语，保持学术风格",

    "模糊2": "简短一些",
    "具体2": "控制在50字以内",

    "模糊3": "格式化输出",
    "具体3": "以Markdown表格形式输出",
}

for key, value in specificity_examples.items():
    print(f"{key}: {value}")

模糊: 写得好一点
具体: 使用专业术语，保持学术风格
模糊2: 简短一些
具体2: 控制在50字以内
模糊3: 格式化输出
具体3: 以Markdown表格形式输出


In [5]:
# 3.3 结构化原则 - 使用分隔符

structured_template = PromptTemplate(
    template="""请分析以下文本的情感倾向：

---
{text}
---

输出格式：
- 情感：正面/负面/中性
- 置信度：0-100
- 关键词：列出3个关键情感词""",
    input_variables=["text"]
)

prompt = structured_template.format(text="这家餐厅的服务态度非常好，菜品也很美味，下次还会再来！")
print(prompt)

请分析以下文本的情感倾向：

---
这家餐厅的服务态度非常好，菜品也很美味，下次还会再来！
---

输出格式：
- 情感：正面/负面/中性
- 置信度：0-100
- 关键词：列出3个关键情感词


## 4. Zero-shot Prompting

**Zero-shot**：不提供任何示例，直接描述任务。

In [6]:
# Zero-shot 示例

zero_shot_examples = {
    "翻译": PromptTemplate(
        template="请将以下英文翻译成中文：\n\n{text}",
        input_variables=["text"]
    ),

    "分类": PromptTemplate(
        template="请将以下新闻标题分类为：体育、科技、娱乐、财经之一。\n\n标题：{title}\n\n类别：",
        input_variables=["title"]
    ),

    "摘要": PromptTemplate(
        template="请用一句话总结以下内容：\n\n{content}\n\n摘要：",
        input_variables=["content"]
    ),
}

# 测试翻译
print("=== 翻译任务 ===")
print(zero_shot_examples["翻译"].format(text="The quick brown fox jumps over the lazy dog."))

print("\n=== 分类任务 ===")
print(zero_shot_examples["分类"].format(title="苹果发布新款iPhone，搭载A18芯片"))

=== 翻译任务 ===
请将以下英文翻译成中文：

The quick brown fox jumps over the lazy dog.

=== 分类任务 ===
请将以下新闻标题分类为：体育、科技、娱乐、财经之一。

标题：苹果发布新款iPhone，搭载A18芯片

类别：


## 5. 角色提示 (Role Prompting)

通过设定角色来引导模型的输出风格和专业度。

In [7]:
# 角色提示模板

class RolePrompts:
    """预定义角色提示"""

    EXPERT = "你是一位拥有20年经验的{domain}专家。"
    TEACHER = "你是一位善于用简单语言解释复杂概念的老师，面向{level}学生。"
    REVIEWER = "你是一位严格的代码评审员，关注{focus}。"
    ASSISTANT = "你是一位友好、耐心的AI助手。"
    TRANSLATOR = "你是一位专业的{source_lang}到{target_lang}翻译。"

# 使用角色提示
role_template = ChatPromptTemplate.from_messages([
    ("system", RolePrompts.EXPERT.format(domain="机器学习")),
    ("user", "请解释什么是梯度下降算法？")
])

print("=== 专家角色 ===")
for msg in role_template.format():
    print(f"{msg['role']}: {msg['content']}")

print("\n=== 教师角色 ===")
teacher_template = ChatPromptTemplate.from_messages([
    ("system", RolePrompts.TEACHER.format(level="初中")),
    ("user", "请解释什么是梯度下降算法？")
])
for msg in teacher_template.format():
    print(f"{msg['role']}: {msg['content']}")

=== 专家角色 ===
system: 你是一位拥有20年经验的机器学习专家。
user: 请解释什么是梯度下降算法？

=== 教师角色 ===
system: 你是一位善于用简单语言解释复杂概念的老师，面向初中学生。
user: 请解释什么是梯度下降算法？


## 6. 输出格式控制

控制模型输出的格式，使其更易于解析和使用。

In [8]:
# JSON 格式输出

json_template = PromptTemplate(
    template="""分析以下产品评论，提取关键信息。

评论：{review}

请以JSON格式返回，包含以下字段：
{{
    "sentiment": "正面/负面/中性",
    "rating": 1-5,
    "pros": ["优点1", "优点2"],
    "cons": ["缺点1", "缺点2"],
    "summary": "一句话总结"
}}

JSON输出：""",
    input_variables=["review"]
)

print(json_template.format(review="这款手机拍照效果很好，电池续航也不错，就是价格有点贵，发热也比较明显。"))

分析以下产品评论，提取关键信息。

评论：这款手机拍照效果很好，电池续航也不错，就是价格有点贵，发热也比较明显。

请以JSON格式返回，包含以下字段：
{
    "sentiment": "正面/负面/中性",
    "rating": 1-5,
    "pros": ["优点1", "优点2"],
    "cons": ["缺点1", "缺点2"],
    "summary": "一句话总结"
}

JSON输出：


In [9]:
# Markdown 格式输出

markdown_template = PromptTemplate(
    template="""请为以下主题生成一份技术报告大纲。

主题：{topic}

请以Markdown格式输出，包含：
- 一级标题（#）
- 二级标题（##）
- 要点列表
- 代码块（如适用）

报告大纲：""",
    input_variables=["topic"]
)

print(markdown_template.format(topic="大语言模型的微调技术"))

请为以下主题生成一份技术报告大纲。

主题：大语言模型的微调技术

请以Markdown格式输出，包含：
- 一级标题（#）
- 二级标题（##）
- 要点列表
- 代码块（如适用）

报告大纲：


In [10]:
# 使用预定义模板库

print("=== 情感分析模板 ===")
print(PromptLibrary.SENTIMENT.format(text="这个产品质量太差了，用了一周就坏了"))

print("\n=== 代码生成模板 ===")
print(PromptLibrary.CODE_GENERATION.format(
    language="python",
    requirement="实现一个二分查找函数"
))

=== 情感分析模板 ===
分析以下文本的情感倾向。

文本：这个产品质量太差了，用了一周就坏了

请以JSON格式输出：
{
    "sentiment": "正面/负面/中性",
    "confidence": 0.0-1.0,
    "reason": "简短解释"
}

分析结果：

=== 代码生成模板 ===
请用python实现以下功能：

需求：实现一个二分查找函数

要求：
- 代码简洁清晰
- 添加必要注释
- 处理边界情况

代码：
```python



## 7. 实战练习

### 练习1：设计一个客服对话提示

In [11]:
# 练习1：客服对话提示

customer_service_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一位专业的电商客服代表。

职责：
- 友好、耐心地回答客户问题
- 提供准确的产品和订单信息
- 处理退换货请求

规则：
- 始终保持礼貌和专业
- 如果不确定，承诺查询后回复
- 每次回复结尾询问是否还有其他问题"""),
    ("user", "{customer_message}")
])

print(customer_service_prompt.format_as_string(
    customer_message="我的订单已经一周了还没收到，能帮我查一下吗？"
))

System: 你是一位专业的电商客服代表。

职责：
- 友好、耐心地回答客户问题
- 提供准确的产品和订单信息
- 处理退换货请求

规则：
- 始终保持礼貌和专业
- 如果不确定，承诺查询后回复
- 每次回复结尾询问是否还有其他问题

User: 我的订单已经一周了还没收到，能帮我查一下吗？


In [12]:
# 练习2：设计一个代码审查提示

code_review_prompt = PromptTemplate(
    template="""作为一位资深代码审查员，请审查以下{language}代码。

代码：
```{language}
{code}
```

请从以下方面进行审查：
1. **代码质量**：可读性、命名规范
2. **性能**：潜在的性能问题
3. **安全性**：安全漏洞
4. **最佳实践**：是否遵循{language}最佳实践

输出格式：
## 审查结果
### 问题列表
- [严重程度] 问题描述
### 改进建议
### 优化后代码""",
    input_variables=["language", "code"]
)

sample_code = '''def get_user(id):
    query = "SELECT * FROM users WHERE id = " + str(id)
    return db.execute(query)'''

print(code_review_prompt.format(language="python", code=sample_code))

作为一位资深代码审查员，请审查以下python代码。

代码：
```python
def get_user(id):
    query = "SELECT * FROM users WHERE id = " + str(id)
    return db.execute(query)
```

请从以下方面进行审查：
1. **代码质量**：可读性、命名规范
2. **性能**：潜在的性能问题
3. **安全性**：安全漏洞
4. **最佳实践**：是否遵循python最佳实践

输出格式：
## 审查结果
### 问题列表
- [严重程度] 问题描述
### 改进建议
### 优化后代码


## 总结

本节我们学习了：

1. **提示工程的基本概念**：通过设计提示引导LLM输出
2. **提示的基本结构**：角色、任务、上下文、示例、格式、约束
3. **设计原则**：清晰性、具体性、结构化
4. **Zero-shot Prompting**：无示例直接描述任务
5. **角色提示**：通过角色设定引导输出风格
6. **输出格式控制**：JSON、Markdown等格式化输出

下一节我们将学习 **Few-shot Learning**，通过提供示例来提升模型表现。